# Manual unit execution for tomography, fidelity, and Frobenius distance

You can edit:
- the ket amplitudes,
- the density matrices,
- the subnormalisation trace factors,
- the tomography shot count.


In [3]:
import numpy as np
import pandas as pd

import os
import sys

project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    
from metric.tomography import (
    perform_single_qubit_tomography,
    normalize_density_matrix,
)
from metric.fidelity_frobenius import (
    evaluate_fidelity_frobenius,
)

In [5]:
# ---------- Helper functions ----------

def ket_to_density(ket: np.ndarray) -> np.ndarray:
    ket = np.asarray(ket, dtype=complex).reshape(-1)
    ket = ket / np.linalg.norm(ket)
    return np.outer(ket, ket.conj())

def make_subnormalised_density_from_ket(alpha, beta, trace_scale=1.0):
    ket = np.array([alpha, beta], dtype=complex)
    rho = ket_to_density(ket)
    return float(trace_scale) * rho

def pretty_matrix(name, mat):
    print(f"{name} =")
    print(np.asarray(mat))
    print("trace =", np.trace(mat))
    print()

def compare_exact_vs_reconstructed(exact_rho, reconstructed_rho, label=""):
    metrics = evaluate_fidelity_frobenius(exact_rho, reconstructed_rho)
    row = {
        "label": label,
        "trace_exact": float(np.real(np.trace(exact_rho))),
        "trace_reconstructed": float(np.real(np.trace(reconstructed_rho))),
        "fidelity": metrics.fidelity,
        "frobenius_distance": metrics.frobenius_distance,
    }
    return row


## 1. Define tweakable states

Edit these states directly.

Notes:
- `state_pure_subnorm` is a **subnormalised pure state**.
- `state_mixed_subnorm` is a **subnormalised mixed state**.
- You can replace any matrix below with your own $2\times 2$ density matrix.


In [6]:
# ---------- Tweakable test states ----------

# Example 1: pure state |psi> = alpha|0> + beta|1|
alpha = np.sqrt(0.70)
beta = np.sqrt(0.30) * np.exp(1j * np.pi / 4)
state_pure = make_subnormalised_density_from_ket(alpha, beta, trace_scale=1.0)

# Example 2: the same pure state, but subnormalised
state_pure_subnorm = make_subnormalised_density_from_ket(alpha, beta, trace_scale=0.62)

# Example 3: a mixed state written directly as a density matrix
state_mixed = np.array([
    [0.65, 0.18 - 0.07j],
    [0.18 + 0.07j, 0.35],
], dtype=complex)

# Example 4: a subnormalised mixed state
state_mixed_subnorm = 0.55 * state_mixed

# Example 5: a custom state you can edit manually
state_custom = np.array([
    [0.88, 0.12 + 0.21j],
    [0.12 - 0.21j, 0.12],
], dtype=complex)

TEST_STATES = {
    "pure_normalised": state_pure,
    "pure_subnormalised": state_pure_subnorm,
    "mixed_normalised": state_mixed,
    "mixed_subnormalised": state_mixed_subnorm,
    "custom_manual": state_custom,
}

for name, rho in TEST_STATES.items():
    pretty_matrix(name, rho)


pure_normalised =
[[0.7       +0.00000000e+00j 0.32403703-3.24037035e-01j]
 [0.32403703+3.24037035e-01j 0.3       +1.16536554e-17j]]
trace = (1+1.165365535093033e-17j)

pure_subnormalised =
[[0.434     +0.00000000e+00j 0.20090296-2.00902962e-01j]
 [0.20090296+2.00902962e-01j 0.186     +7.22526632e-18j]]
trace = (0.6200000000000001+7.225266317576804e-18j)

mixed_normalised =
[[0.65+0.j   0.18-0.07j]
 [0.18+0.07j 0.35+0.j  ]]
trace = (1+0j)

mixed_subnormalised =
[[0.3575+0.j     0.099 -0.0385j]
 [0.099 +0.0385j 0.1925+0.j    ]]
trace = (0.55+0j)

custom_manual =
[[0.88+0.j   0.12+0.21j]
 [0.12-0.21j 0.12+0.j  ]]
trace = (1+0j)



## 2. Tomography on one chosen state

Change:
- `selected_label`
- `shots`

Set `shots = None` for exact tomography from exact probabilities.


In [7]:
selected_label = "pure_subnormalised"   # change this
shots = 1024                              # set to None for exact tomography
seed = 12345

rho_exact = TEST_STATES[selected_label]

tomography_result = perform_single_qubit_tomography(
    rho_exact,
    shots=shots,
    seed=seed,
    enforce_physical=True,
)

pretty_matrix("exact input rho", tomography_result.input_density_matrix)
pretty_matrix("linear inversion rho", tomography_result.reconstructed_density_matrix)
pretty_matrix("physical projected rho", tomography_result.physical_density_matrix)

print("Ideal expectations:", tomography_result.ideal_expectations)
print("Measured expectations:", tomography_result.measured_expectations)
print("Basis probabilities:", tomography_result.basis_probabilities)
print("Counts:", tomography_result.counts)


exact input rho =
[[0.434     +0.00000000e+00j 0.20090296-2.00902962e-01j]
 [0.20090296+2.00902962e-01j 0.186     +7.22526632e-18j]]
trace = (0.6200000000000001+7.225266317576804e-18j)

linear inversion rho =
[[0.42201172+0.j         0.20041016-0.19677734j]
 [0.20041016+0.19677734j 0.19798828+0.j        ]]
trace = (0.6200000000000001+0j)

physical projected rho =
[[0.42201172+0.j         0.20041016-0.19677734j]
 [0.20041016+0.19677734j 0.19798828+0.j        ]]
trace = (0.6200000000000003+0j)

Ideal expectations: {'X': 0.648074069840786, 'Y': 0.648074069840786, 'Z': 0.39999999999999997}
Measured expectations: {'X': 0.646484375, 'Y': 0.634765625, 'Z': 0.361328125}
Basis probabilities: {'X': {'0': 0.8240370349203929, '1': 0.17596296507960701}, 'Y': {'0': 0.8240370349203929, '1': 0.17596296507960701}, 'Z': {'0': 0.7, '1': 0.3}}
Counts: {'X': {'0': 843, '1': 181}, 'Y': {'0': 837, '1': 187}, 'Z': {'0': 697, '1': 327}}


## 3. Fidelity and Frobenius distance for the chosen state

This compares the exact target state with:
- the raw linear-inversion tomography state,
- the physical projected tomography state.


In [8]:
metrics_raw = evaluate_fidelity_frobenius(
    tomography_result.input_density_matrix,
    tomography_result.reconstructed_density_matrix,
)

metrics_physical = evaluate_fidelity_frobenius(
    tomography_result.input_density_matrix,
    tomography_result.physical_density_matrix,
)

comparison_df = pd.DataFrame([
    {
        "reconstruction": "linear_inversion",
        "fidelity": metrics_raw.fidelity,
        "frobenius_distance": metrics_raw.frobenius_distance,
    },
    {
        "reconstruction": "physical_projected",
        "fidelity": metrics_physical.fidelity,
        "frobenius_distance": metrics_physical.frobenius_distance,
    },
])

comparison_df


,reconstruction,fidelity,frobenius_distance
0,linear_inversion,0.987438,0.017943
1,physical_projected,0.987438,0.017943


## 4. Examine all manual states at once

This lets you compare how tomography behaves across several manually chosen states.


In [9]:
shots_all = 2048   # change this
seed_base = 2026

rows = []
for i, (label, rho) in enumerate(TEST_STATES.items()):
    tomo = perform_single_qubit_tomography(
        rho,
        shots=shots_all,
        seed=seed_base + i,
        enforce_physical=True,
    )
    rows.append(compare_exact_vs_reconstructed(
        tomo.input_density_matrix,
        tomo.reconstructed_density_matrix,
        label=label + " | raw",
    ))
    rows.append(compare_exact_vs_reconstructed(
        tomo.input_density_matrix,
        tomo.physical_density_matrix,
        label=label + " | physical",
    ))

summary_df = pd.DataFrame(rows)
summary_df


,label,trace_exact,trace_reconstructed,fidelity,frobenius_distance
0,pure_normalised | raw,1.00,1.00,1.003290,0.014575
1,pure_normalised | physical,1.00,1.00,0.999906,0.013720
2,pure_subnormalised | raw,0.62,0.62,1.009127,0.008226
3,pure_subnormalised | physical,0.62,0.62,0.999995,0.001870
4,mixed_normalised | raw,1.00,1.00,0.999851,0.017224
5,mixed_normalised | physical,1.00,1.00,0.999851,0.017224
6,mixed_subnormalised | raw,0.55,0.55,0.999858,0.009268
7,mixed_subnormalised | physical,0.55,0.55,0.999858,0.009268
8,custom_manual | raw,1.00,1.00,0.999914,0.009978
9,custom_manual | physical,1.00,1.00,0.999914,0.009978


## 5. Direct pairwise fidelity/Frobenius tests without tomography

Use this section when you want to compare **two states you choose manually**.


In [10]:
# Pairwise manual comparison
rho_A = TEST_STATES["pure_subnormalised"]      # change this
rho_B = TEST_STATES["mixed_subnormalised"]     # change this

pairwise_metrics = evaluate_fidelity_frobenius(rho_A, rho_B)

pretty_matrix("rho_A", rho_A)
pretty_matrix("rho_B", rho_B)

print("Pairwise fidelity =", pairwise_metrics.fidelity)
print("Pairwise Frobenius distance =", pairwise_metrics.frobenius_distance)


rho_A =
[[0.434     +0.00000000e+00j 0.20090296-2.00902962e-01j]
 [0.20090296+2.00902962e-01j 0.186     +7.22526632e-18j]]
trace = (0.6200000000000001+7.225266317576804e-18j)

rho_B =
[[0.3575+0.j     0.099 -0.0385j]
 [0.099 +0.0385j 0.1925+0.j    ]]
trace = (0.55+0j)

Pairwise fidelity = 0.7220185174601967
Pairwise Frobenius distance = 0.2818020068987196


## 6. Blank template for your own state

Edit the entries below directly. Make sure the matrix is Hermitian and positive semidefinite.


In [12]:
# ---------- Blank editable template ----------

rho_user = np.array([
    [1.50, 0.20 + 0.05j],
    [0.20 - 0.05j, 0.50],
], dtype=complex)

pretty_matrix("rho_user", rho_user)

tomo_user = perform_single_qubit_tomography(
    rho_user,
    shots=4096,      # change freely
    seed=7,
    enforce_physical=True,
)

user_metrics = evaluate_fidelity_frobenius(
    rho_user,
    tomo_user.physical_density_matrix,
)

pretty_matrix("rho_user reconstructed", tomo_user.physical_density_matrix)
print("User-state fidelity =", user_metrics.fidelity)
print("User-state Frobenius distance =", user_metrics.frobenius_distance)


rho_user =
[[1.5+0.j   0.2+0.05j]
 [0.2-0.05j 0.5+0.j  ]]
trace = (2+0j)

rho_user reconstructed =
[[1.51269531+0.j         0.20166016+0.07128906j]
 [0.20166016-0.07128906j 0.48730469+0.j        ]]
trace = (1.9999999999999996+0j)

User-state fidelity = 0.999822533744745
User-state Frobenius distance = 0.035132641812877737
